## Objetivo

Para la predicción financiera se han usado modelos que han estado presente por mucho tiempo en el mercado (ARIMA, SARIMA, GARCH, etc.). A estos modelos se les conoce como *Pronósticos de Series de Tiempo* (Time Series Forecasting) dado que tratan con historial de datos donde el tiempo les da su estructura. Estos modelos analizan las tendencias, la estacionalidad y el ruido para aprender los patrones inherentes en el tiempo y hacer la mejor estimación.

Sin embargo, podemos transformar los datos históricos de tal forma que funcione como entrenamiento para nuestros modelos de aprendizaje supervizado y esta transformación de la data se le conoce como *Sliding Window*.

En éste Juppyter Notebook presentaré una manera de usar *Sliding Window* para crear la **Tabla Analítica de Datos** que servirá para alimentar a nuestros modelos. Para esto definiré lo siguiente:

* Ventadas de Observación (vobs): El pasado que el modelo puede ver para aprender patrones. En este caso será de 15 días.
* Ventana de Desempeño (vdes): Es el futuro que quieres predecir. Esta será la variable objetivo. En este caso 1 día.
* Step (stride): Es cuánto se desplaza la ventana cada vez. Controla los pasos que genera y define cuánto se superponen las ventanas. En este caso 3 días de desplazamiento. 

## Dependencias


In [11]:
import yfinance as yf
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit


from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import cufflinks as cf
import plotly.express as px


from functools import reduce
from dateutil.relativedelta import relativedelta as rd

import warnings
import os 

warnings.filterwarnings('ignore')
cf.go_offline()
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', '{:,.4f}'.format)

## Lectura de Datos

In [12]:
df = pd.read_csv("df.csv")

### Catálogo de Fechas

Crearemos un catálogo de fechas para poder trabajar con más facilidad.

In [13]:
# Definimos las Ventanas de Observación, Ventanas de Desempeño y los pasos que darán

vobs = 15
vdes = 1
step = 3

In [14]:
cat_date = pd.DataFrame(df['Date'].unique(), columns=['Date']).sort_values(['Date'])
cat_date.insert(0,'id', cat_date.index +1)

In [15]:
df = df.merge(
    cat_date,
    on='Date',
    how='left',
)

df.head()

,Date,Close,High,Low,Open,Volume,sma_5,sma_10,sma_20,ema_10,ema_20,dist_sma10,sma_cross,retorno_1,retorno_5,retorno_10,retorno_20,mom_10,mom_20,risk_5,risk_10,risk_20,range,range_roll_14,rsi_14,macd,macd_signal,macd_hist,day,month,roll_max_20,roll_min_20,dist_resistance,dist_support,vol_sma_10,vol_ratio,id
0,2024-03-11,56.3918,56.4386,55.7739,55.9612,14114300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6646,NaN,NaN,0.0000,0.0000,0.0000,0,3,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2024-03-12,56.6352,56.8692,56.3637,56.4667,12684600,NaN,NaN,NaN,56.3918,56.3918,NaN,NaN,0.0043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5055,NaN,NaN,0.0055,0.0030,0.0024,1,3,NaN,NaN,NaN,NaN,NaN,NaN,2
2,2024-03-13,57.2155,57.2998,56.9160,56.9909,13909500,NaN,NaN,NaN,56.5256,56.5195,NaN,NaN,0.0102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3838,NaN,NaN,0.0250,0.0120,0.0129,2,3,NaN,NaN,NaN,NaN,NaN,NaN,3
3,2024-03-14,57.0882,57.3713,56.9938,57.1637,13996600,NaN,NaN,NaN,56.8030,56.7751,NaN,NaN,-0.0022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3774,NaN,NaN,0.0286,0.0176,0.0110,3,3,NaN,NaN,NaN,NaN,NaN,NaN,4
4,2024-03-15,56.5031,57.0410,56.2767,56.6352,36849900,NaN,NaN,NaN,56.8969,56.8655,NaN,NaN,-0.0102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.7643,NaN,NaN,0.0067,0.0144,-0.0077,4,3,NaN,NaN,NaN,NaN,NaN,NaN,5


## Matriz de Predictoras (Sliding Windows)

Para crear nuestra Tabla Analítica de Datos que alimentará a los modelos, usaremos *Sliding Windows* y usaremos las siguientes operaciones como *Feature Engineering*:

* Media
* Mediana
* Desviación Estándard
* Mínimo
* Máximo

Pero antes, necesitamos definir la primera ancla y la última amcla que sean *válidas* (i.e. que tengan suficiente data para los pasos que estoy dando).

In [16]:
anclai, anclaf = df['id'].min() + vobs - 1, df['id'].max() - vdes

print(f"La primera ancla es {anclai} y la ancla final será {anclaf}")
print(f"Se tiene {len(list(range(anclai, anclaf+1, step)))} anclas para trabajar")

La primera ancla es 15 y la ancla final será 499
Se tiene 162 anclas para trabajar


In [17]:
# Definimos las funciones que aplicaremos para feature extraction y la unidad muestral

um = ['ancla']

func = ['mean','median','std','min','max']
funcName = [np.mean, np.median, np.std, np.min, np.max]

varc = ['sma_5',
       'sma_10', 'sma_20', 'ema_10', 'ema_20', 'dist_sma10', 'sma_cross',
       'retorno_1', 'retorno_5', 'retorno_10', 'retorno_20', 'mom_10',
       'mom_20', 'risk_5', 'risk_10', 'risk_20', 'range', 'range_roll_14',
       'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'day', 'month',
       'roll_max_20', 'roll_min_20', 'dist_resistance', 'dist_support',
       'vol_sma_10', 'vol_ratio']

In [18]:
ancla = 15
sub = 3
def ingX(df:pd.DataFrame, ancla:int, sub:int)->pd.DataFrame:
    aux = df.loc[(df['id']<=ancla)&(df['id']>=ancla - sub +1)]

    feats = {}

    for c in varc:
        serie = aux[c]
        
        for nombre,funcion in zip(func,funcName):
            
            feats[f'x_{c}_{nombre}_{sub}'] = funcion(serie)
            
    out = pd.DataFrame([feats])
    out.insert(0,'ancla',ancla)
    return out

In [19]:
# Funciones para poder aplicar la fiunción para todas las anclas y todas las sub anclas
# dado que el resultado serán otros dataframes

apilar = lambda x,y: pd.concat([x,y], ignore_index=True)
cruzar = lambda x,y: x.merge(y, on=um, how='outer')

In [20]:
X = reduce(
    cruzar,
    map(
        lambda sub: reduce(
            apilar,
            map(lambda ancla: ingX(df, ancla, sub), range(anclai, anclaf+1, step))
        ),
        range(step, step+vobs, step)
    )
)

## Vector de predicción

Para obtener el vector de predicción, solo pasaremos el valor de $t+1$ como un valor a predecir en el tiempo $t$. Vean el siguiente ejemplo:

In [21]:
def ingY(df:pd.DataFrame, ancla:int)->pd.DataFrame:
    future = df.loc[df['id'] == ancla + vdes, 'Close']
    
    if len(future) == 0:
        return pd.DataFrame()
    
    y_val = future.values[0] / df.loc[df['id']==ancla, 'Close'].values[0] - 1
    
    return pd.DataFrame({
        'ancla': [ancla],
        'y': [y_val]
    })

In [22]:
y = reduce(
    apilar,
    map(lambda ancla: ingY(df, ancla), range(anclai, anclaf+1, step))
)

In [23]:
tad = X.merge(y, on='ancla',how='inner')

In [24]:
tad = tad.dropna().reset_index(drop=True)
tad.head()

,ancla,x_sma_5_mean_3,x_sma_5_median_3,x_sma_5_std_3,x_sma_5_min_3,x_sma_5_max_3,x_sma_10_mean_3,x_sma_10_median_3,x_sma_10_std_3,x_sma_10_min_3,x_sma_10_max_3,x_sma_20_mean_3,x_sma_20_median_3,x_sma_20_std_3,x_sma_20_min_3,x_sma_20_max_3,x_ema_10_mean_3,x_ema_10_median_3,x_ema_10_std_3,x_ema_10_min_3,x_ema_10_max_3,x_ema_20_mean_3,x_ema_20_median_3,x_ema_20_std_3,x_ema_20_min_3,x_ema_20_max_3,x_dist_sma10_mean_3,x_dist_sma10_median_3,x_dist_sma10_std_3,x_dist_sma10_min_3,x_dist_sma10_max_3,x_sma_cross_mean_3,x_sma_cross_median_3,x_sma_cross_std_3,x_sma_cross_min_3,x_sma_cross_max_3,x_retorno_1_mean_3,x_retorno_1_median_3,x_retorno_1_std_3,x_retorno_1_min_3,x_retorno_1_max_3,x_retorno_5_mean_3,x_retorno_5_median_3,x_retorno_5_std_3,x_retorno_5_min_3,x_retorno_5_max_3,x_retorno_10_mean_3,x_retorno_10_median_3,x_retorno_10_std_3,x_retorno_10_min_3,x_retorno_10_max_3,x_retorno_20_mean_3,x_retorno_20_median_3,x_retorno_20_std_3,x_retorno_20_min_3,x_retorno_20_max_3,x_mom_10_mean_3,x_mom_10_median_3,x_mom_10_std_3,x_mom_10_min_3,x_mom_10_max_3,x_mom_20_mean_3,x_mom_20_median_3,x_mom_20_std_3,x_mom_20_min_3,x_mom_20_max_3,x_risk_5_mean_3,x_risk_5_median_3,x_risk_5_std_3,x_risk_5_min_3,x_risk_5_max_3,x_risk_10_mean_3,x_risk_10_median_3,x_risk_10_std_3,x_risk_10_min_3,x_risk_10_max_3,x_risk_20_mean_3,x_risk_20_median_3,x_risk_20_std_3,x_risk_20_min_3,x_risk_20_max_3,x_range_mean_3,x_range_median_3,x_range_std_3,x_range_min_3,x_range_max_3,x_range_roll_14_mean_3,x_range_roll_14_median_3,x_range_roll_14_std_3,x_range_roll_14_min_3,x_range_roll_14_max_3,x_rsi_14_mean_3,x_rsi_14_median_3,x_rsi_14_std_3,x_rsi_14_min_3,x_rsi_14_max_3,x_macd_mean_3,x_macd_median_3,x_macd_std_3,x_macd_min_3,...,x_retorno_20_median_15,x_retorno_20_std_15,x_retorno_20_min_15,x_retorno_20_max_15,x_mom_10_mean_15,x_mom_10_median_15,x_mom_10_std_15,x_mom_10_min_15,x_mom_10_max_15,x_mom_20_mean_15,x_mom_20_median_15,x_mom_20_std_15,x_mom_20_min_15,x_mom_20_max_15,x_risk_5_mean_15,x_risk_5_median_15,x_risk_5_std_15,x_risk_5_min_15,x_risk_5_max_15,x_risk_10_mean_15,x_risk_10_median_15,x_risk_10_std_15,x_risk_10_min_15,x_risk_10_max_15,x_risk_20_mean_15,x_risk_20_median_15,x_risk_20_std_15,x_risk_20_min_15,x_risk_20_max_15,x_range_mean_15,x_range_median_15,x_range_std_15,x_range_min_15,x_range_max_15,x_range_roll_14_mean_15,x_range_roll_14_median_15,x_range_roll_14_std_15,x_range_roll_14_min_15,x_range_roll_14_max_15,x_rsi_14_mean_15,x_rsi_14_median_15,x_rsi_14_std_15,x_rsi_14_min_15,x_rsi_14_max_15,x_macd_mean_15,x_macd_median_15,x_macd_std_15,x_macd_min_15,x_macd_max_15,x_macd_signal_mean_15,x_macd_signal_median_15,x_macd_signal_std_15,x_macd_signal_min_15,x_macd_signal_max_15,x_macd_hist_mean_15,x_macd_hist_median_15,x_macd_hist_std_15,x_macd_hist_min_15,x_macd_hist_max_15,x_day_mean_15,x_day_median_15,x_day_std_15,x_day_min_15,x_day_max_15,x_month_mean_15,x_month_median_15,x_month_std_15,x_month_min_15,x_month_max_15,x_roll_max_20_mean_15,x_roll_max_20_median_15,x_roll_max_20_std_15,x_roll_max_20_min_15,x_roll_max_20_max_15,x_roll_min_20_mean_15,x_roll_min_20_median_15,x_roll_min_20_std_15,x_roll_min_20_min_15,x_roll_min_20_max_15,x_dist_resistance_mean_15,x_dist_resistance_median_15,x_dist_resistance_std_15,x_dist_resistance_min_15,x_dist_resistance_max_15,x_dist_support_mean_15,x_dist_support_median_15,x_dist_support_std_15,x_dist_support_min_15,x_dist_support_max_15,x_vol_sma_10_mean_15,x_vol_sma_10_median_15,x_vol_sma_10_std_15,x_vol_sma_10_min_15,x_vol_sma_10_max_15,x_vol_ratio_mean_15,x_vol_ratio_median_15,x_vol_ratio_std_15,x_vol_ratio_min_15,x_vol_ratio_max_15,y
0,57,58.8181,58.8470,0.1524,58.6187,58.9886,59.1726,59.1613,0.1220,59.0292,59.3273,59.0896,59.0872,0.0058,59.0839,59.0976,58.9363,58.9350,0.1076,58.8051,59.0687,58.7885,58.7905,0.0420,58.7361,58.8388,-0.0140,-0.0159,0.0033,-0.0167,-0.0094,-0.2714,-0.2401,0.1471,-0.4652,-0.1090,-0.0002,-0.0019,0.0032,-0.0029,0.0044,-0.0159,-0.0163,0.0030,-0.0192,-0.0120,-0.0227,-0.0222,0.0038,-0.0277,-0.0184,-0.0013

## Partición

La partición se hará **Off Time**, es decir, sacamos el 10% del nuestros datos que no verá el modelo. Para los demás conjuntos se utiliza el quantile 0.85 para dividir en train y test.

In [25]:
cut_off = int(tad['ancla'].quantile(0.90))

Sn = tad[tad['ancla'] < cut_off].reset_index(drop=True)
ot = tad[tad['ancla'] >= cut_off].reset_index(drop=True)

cut_ancla = int(Sn['ancla'].quantile(0.85))

train_df = Sn[Sn['ancla'] < cut_ancla].reset_index(drop=True)
test_df  = Sn[Sn['ancla'] >= cut_ancla].reset_index(drop=True)

## EDA

In [26]:
# Revisamos los missings

miss = train_df.isna().sum().reset_index(name='miss').sort_values('miss', ascending=False).reset_index(drop=True)
miss['miss'] = miss['miss'] / len(Sn)
miss

,index,miss
0,ancla,0.0000
1,x_sma_5_mean_3,0.0000
2,x_sma_5_median_3,0.0000
3,x_sma_5_std_3,0.0000
4,x_sma_5_min_3,0.0000
...,...,...
747,x_vol_ratio_median_15,0.0000
748,x_vol_ratio_std_15,0.0000
749,x_vol_ratio_min_15,0.0000
750,x_vol_ratio_max_15,0.0000


In [27]:
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
test_df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [28]:
# Revisamos infs
np.isinf(train_df).sum().reset_index(name='infs').sort_values('infs', ascending=False).reset_index(drop=True)

,index,infs
0,ancla,0
1,x_sma_5_mean_3,0
2,x_sma_5_median_3,0
3,x_sma_5_std_3,0
4,x_sma_5_min_3,0
...,...,...
747,x_vol_ratio_median_15,0
748,x_vol_ratio_std_15,0
749,x_vol_ratio_min_15,0
750,x_vol_ratio_max_15,0


In [29]:
tgt = ['y']

In [30]:
varc = [c for c in train_df.columns if c.startswith('x_')]

imputer = SimpleImputer(strategy='median')

X_train_imp = pd.DataFrame(
    imputer.fit_transform(train_df[varc]),
    columns=varc,
    index=train_df.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(test_df[varc]),
    columns=varc,
    index=test_df.index
)